# Skala in PyFock: energies, forces and geometry optimization

[Skala](https://github.com/microsoft/skala) is a neural exchange-correlation functional from Microsoft
Research AI for Science. In PyFock it is selected by name, like any other functional.

This notebook covers:

1. a single-point energy,
2. analytical nuclear forces,
3. a geometry optimization with ASE's `LBFGSLineSearch`,
4. reproducing Skala's own published reference energies, with a parity plot and RMSE / MAE / R².

PyFock reads Skala's published TorchScript checkpoint directly, so the `skala` package — and with it
PySCF — is **not** required. That is also why this works on Windows.

In [ ]:
pip install -q "pyfock[skala,ase]" matplotlib

In [ ]:
import contextlib, io, json, pathlib, urllib.request
import numpy as np
from pyfock import Basis, Data, DFT, DFT_Grad, Grids, Mol, XC

NCORES = 4

def quiet(func, *args, **kwargs):
    """Run something with PyFock's (very verbose) SCF log suppressed."""
    with contextlib.redirect_stdout(io.StringIO()):
        return func(*args, **kwargs)

assert hasattr(XC, 'is_skala'), 'this PyFock build has no Skala support — see the install cell above'
print('Skala functionals available:', sorted(XC.SKALA_FUNCTIONALS))

## 1. Single-point energy

Skala is parametrised together with a DFT-D3 correction, so `dispersion=True` is what you want for real
chemistry. The correction is additive and does not enter the SCF.

In [ ]:
mol      = Mol(atoms=[['O', 0.0, 0.0, 0.1173], ['H', 0.0, 0.7572, -0.4692], ['H', 0.0, -0.7572, -0.4692]])
basis    = Basis(mol, {'all': Basis.load(mol=mol, basis_name='def2-SVP')})
auxbasis = Basis(mol, {'all': Basis.load(mol=mol, basis_name='def2-universal-jfit')})

dft = DFT(mol, basis, auxbasis, xc='skala-1.1', grids=Grids(mol, level=3, verbose=False),
          dispersion=True)
dft.conv_crit, dft.ncores = 1e-8, NCORES

energy, dmat = quiet(dft.scf)
print(f'E(Skala + D3) = {energy:.8f} Ha   in {dft.niter} iterations')
print(f'  of which D3 = {dft.Edisp:.8f} Ha')

## 2. Analytical forces

`DFT_Grad` works with Skala exactly as with a semilocal functional. The XC gradient carries the full
grid response — the Becke weight derivatives and the grid-translation term — which a non-local
functional needs and which PyFock omits for semilocal ones. The net force below should vanish.

In [ ]:
from pyfock import DFT_Grad

forces = quiet(DFT_Grad(dft, verbose=False).calculate)['forces']
print('forces (Ha/Bohr):')
print(np.round(forces, 6))
print('net force (translational invariance):', np.abs(forces.sum(0)).max())

## 3. Geometry optimization with ASE

`PyFockCalculator` hands those analytical forces to ASE, so any ASE optimizer works. The D3 term comes
through the calculator's own dispersion hook, with Skala's `b3lyp5` parameters.

In [ ]:
from ase import Atoms
from ase.optimize import LBFGSLineSearch
from pyfock import PyFockCalculator

water = Atoms('OHH', positions=[[0.0, 0.0, 0.12], [0.0, 0.80, -0.48], [0.0, -0.80, -0.48]])
water.calc = PyFockCalculator(functional='skala-1.1', basis='def2-SVP',
                              auxbasis='def2-universal-jfit', ncores=NCORES, conv_crit=1e-8,
                              dispersion=True, dispersion_kwargs={'xc': 'b3lyp5'})

quiet(LBFGSLineSearch(water, logfile=None).run, fmax=0.02)

print(f'O-H = {water.get_distance(0, 1):.4f}, {water.get_distance(0, 2):.4f} A')
print(f'HOH = {water.get_angle(1, 0, 2):.2f} deg')

## 4. Reproducing Skala's published reference energies

The Skala repository ships the total energies behind its benchmark report as
`benchmark/reference/measurements.json`. The geometries are not redistributed; they come from
`grimme-lab/GMTKN55` at a pinned commit.

Their protocol has to be matched exactly:

| setting | value |
|---|---|
| orbitals | spherical → `dft.sao = True` |
| density fitting | `def2-universal-jkfit` |
| grid | level 3 |
| convergence | `conv_crit = 5e-6` |
| dispersion | **included** — `SkalaKS` defaults to `with_dftd3=True`, so their numbers carry the D3 term |

In [ ]:
CACHE = pathlib.Path('skala_ref_cache'); CACHE.mkdir(exist_ok=True)
REF_URL = ('https://media.githubusercontent.com/media/microsoft/skala/main/'
           'benchmark/reference/measurements.json')
GMTKN55 = ('https://raw.githubusercontent.com/grimme-lab/GMTKN55/'
           '8d485b37a1ca8837e395042671ca5ba4e0714691/')

def fetch(url, name):
    path = CACHE / name
    if not path.exists():
        path.write_bytes(urllib.request.urlopen(url, timeout=300).read())
    return path

# Reference energies, averaged over the repeated runs of each configuration.
reference = {}
for r in json.loads(fetch(REF_URL, 'measurements.json').read_text()):
    if r.get('status') == 'ok' and r.get('total_energy') is not None:
        reference.setdefault((r['mol_name'], r['functional'], r['basis']), []).append(r['total_energy'])
reference = {k: float(np.mean(v)) for k, v in reference.items()}
print(len(reference), 'reference configurations loaded')

def load_geometry(path):
    scale, atoms, inside = 1.0 / Data.Angs2BohrFactor, [], False   # coord files are in Bohr
    for line in fetch(GMTKN55 + path, path.replace('/', '_')).read_text().splitlines():
        token = line.strip()
        if token.startswith('$coord'):
            inside = True
        elif token.startswith('$'):
            inside = False
        elif inside and token:
            x, y, z, symbol = token.split()[:4]
            atoms.append([symbol.capitalize(), float(x) * scale, float(y) * scale, float(z) * scale])
    return Mol(atoms=atoms)

In [ ]:
# The smallest entries of the benchmark set, in order of size.
MOLECULES = [
    ('H2',      'W4-11/h2/coord'),           ('H2O',   'W4-11/h2o/coord'),
    ('C2N2',    'W4-11/nccn/coord'),         ('CHNO',  'W4-11/hnco/coord'),
    ('H2O2',    'W4-11/hooh/coord'),         ('H3N',   'W4-11/nh3/coord'),
    ('CH2N4',   'TAUT15/7a/coord'),          ('H5N3',  'ICONF/N3H5_2/coord'),
    ('C4H3NO2', 'DARC/maleinNH/coord'),      ('C4H5N', 'BHPERI/05r/coord'),
    ('C5H10O',  'FH51/propyloxirane/coord'), ('C8H7N', 'S22/21a/coord'),
]
BASIS = 'def2-svp'    # the reference records use lowercase basis names

def skala_energy(mol):
    basis = Basis(mol, {'all': Basis.load(mol=mol, basis_name=BASIS)})
    aux   = Basis(mol, {'all': Basis.load(mol=mol, basis_name='def2-universal-jkfit')})
    dft = DFT(mol, basis, aux, xc='skala-1.1', grids=Grids(mol, level=3, verbose=False),
              dispersion=True)
    dft.sao, dft.conv_crit, dft.ncores = True, 5e-6, NCORES
    return float(quiet(dft.scf)[0])

names, ours, theirs = [], [], []
for name, path in MOLECULES:
    key = (name, 'skala-1.1', BASIS)
    if key not in reference:
        print(f'{name:>9}  not in the reference set, skipping'); continue
    energy = skala_energy(load_geometry(path))
    names.append(name); ours.append(energy); theirs.append(reference[key])
    print(f'{name:>9}  PyFock {energy:15.8f}   reference {reference[key]:15.8f}   '
          f'diff {energy - reference[key]:+.2e} Ha')

ours, theirs = np.array(ours), np.array(theirs)

In [ ]:
import matplotlib.pyplot as plt

residual = ours - theirs
rmse = float(np.sqrt((residual ** 2).mean()))
mae  = float(np.abs(residual).mean())
r2   = 1.0 - (residual ** 2).sum() / ((theirs - theirs.mean()) ** 2).sum()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

lo, hi = theirs.min(), theirs.max()
pad = 0.03 * (hi - lo)
ax1.plot([lo - pad, hi + pad], [lo - pad, hi + pad], '-', lw=1, color='0.6', zorder=1)
ax1.scatter(theirs, ours, s=45, zorder=2, edgecolor='k', linewidth=0.5)
ax1.set_xlabel('Skala reference (Ha)'); ax1.set_ylabel('PyFock (Ha)')
ax1.set_title('Total energy, skala-1.1 / def2-SVP')
ax1.text(0.04, 0.95,
         f'RMSE = {rmse * 1e6:8.4f} uHa\nMAE  = {mae * 1e6:8.4f} uHa\n'
         f'R2   = {r2:.12f}\nN    = {len(ours)}',
         transform=ax1.transAxes, va='top', family='monospace', fontsize=9)

ax2.axhline(0, lw=1, color='0.6')
ax2.scatter(range(len(residual)), residual * 1e6, s=45, edgecolor='k', linewidth=0.5)
ax2.set_xticks(range(len(names))); ax2.set_xticklabels(names, rotation=60, ha='right', fontsize=8)
ax2.set_ylabel('PyFock - reference (uHa)'); ax2.set_title('Residuals')

fig.tight_layout(); plt.show()

print(f'RMSE {rmse:.3e} Ha    MAE {mae:.3e} Ha    R^2 {r2:.12f}    N = {len(ours)}')

The residuals sit at the level of the reference data's own run-to-run spread — a few times
$10^{-8}$ Ha, on total energies spanning hundreds of Hartree. R² is 1 to twelve digits and is
uninformative at that spread; the residual panel is the meaningful one.

Two things are easy to get wrong, and both show up immediately as a systematic offset:

- **leaving dispersion off** — the published Skala numbers include the D3 term, so a bare energy is off
  by exactly the dispersion energy (5.7e-4 Ha for H₂O);
- **leaving `sao = False`** — the reference uses spherical orbitals, while PyFock defaults to Cartesian.